# 02_07_Profiling and Cleaning `population.xlsx`

In [1]:
from pathlib import Path

import pandas as pd

In [2]:
PATH_RAW = Path("../data/raw/")
PATH_INTERMEDIATE = Path("../data/intermediate/")

# Table of Contents

- [7. POPULATION](#7-population)
- [7.1. Data Profiling](#71-data-profiling)
    - [7.1.1. Overview](#711-overview)

- [7.2. Data Preparation](#72-data-preparation)
    - [7.2.1. Renaming Columns](#721-renaming-columns)
    - [7.2.2. Rounding `area` and `population_density`](#722-rounding-area-and-population_density)

- [7.3. Export to Silver Layer](#73-export-to-silver-layer)

## 7. POPULATION

The `population.xlsx` is downloaded from the Statbel Open Data portal.

It will be used to enrich the `stations_dimension` with municipal population and density area, enabling punctuality analyses at station level filtered by population density categories such as urban or rural areas.

## 7.1. Data Profiling

### 7.1.1. Overview

* The first row of the Excel sheet is the table title, not the column headers. The actual column headers are on the second row - therefore `header=1`.

* The dataset has neither missing values nor duplicates.

* The data types matches the expected types:  
    * `str` for the territorial division names
    * `int64` for the Refnis code and the population count
    * `float64` for the area and the population density

* `Refnis` only contains unique values. 

* Once this dataset has been cleaned, it will be merged with `municipalities_cleaned` on the `Refnis` column.

In [3]:
population = pd.read_excel(PATH_RAW / "population.xlsx", header=1)
population.head()

,Refnis,Lieu de résidence,Population,Superficie au km²,Habitants / km²
0,1000,Belgique,11431406,30689.368862,372.487491
1,2000,Région flamande,6589069,13625.548930,483.581912
2,3000,Région wallonne,3633795,16901.400088,214.999644
3,4000,Région de Bruxelles-Capitale,1208542,162.419844,7440.851870
4,10000,Province d’Anvers,1857986,2876.120363,646.004258


In [4]:
population.info()

<class 'pandas.DataFrame'>
RangeIndex: 638 entries, 0 to 637
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Refnis             638 non-null    int64  
 1   Lieu de résidence  638 non-null    str    
 2   Population         638 non-null    int64  
 3   Superficie au km²  638 non-null    float64
 4   Habitants / km²    638 non-null    float64
dtypes: float64(2), int64(2), str(1)
memory usage: 31.6 KB


In [5]:
population.isnull().sum()

Refnis               0
Lieu de résidence    0
Population           0
Superficie au km²    0
Habitants / km²      0
dtype: int64

In [6]:
population.nunique()

Refnis               638
Lieu de résidence    638
Population           633
Superficie au km²    636
Habitants / km²      636
dtype: int64

In [7]:
population.duplicated().sum()

np.int64(0)

In [8]:
population["Refnis"].duplicated().sum()

np.int64(0)

## 7.2. Data Preparation

### 7.2.1. Renaming Columns

In [9]:
population.columns

Index(['Refnis', 'Lieu de résidence', 'Population', 'Superficie au km²',
       'Habitants / km²'],
      dtype='str')

To avoid issues with the `"²"` and the `"/"` special characters, and to translate their names to English, the columns are renamed in `snake_case`.

In [10]:
population = population.rename(columns={
                                    "Refnis" : "refnis",
                                    "Lieu de résidence" : "place_of_residence",
                                    "Population" : "population",
                                    "Superficie au km²" : "area",
                                    "Habitants / km²" : "population_density"
                                    }
                                )

In [11]:
population.columns

Index(['refnis', 'place_of_residence', 'population', 'area',
       'population_density'],
      dtype='str')

### 7.2.2. Rounding `area` and `population_density`

The `area` and `population_density` values in the original Statbel dataset are provided with six decimal places, which exceeds the precision required for this analysis. Both columns are rounded to **two decimal places**, as the intended use of `population_density` is to create a derived categorical column based on official Belgian population density category thresholds, and to filter Power BI dashboards by **population density categories** rather than to perform precise numerical calculations.

In [12]:
population["population_density"] = population["population_density"].round(2)
population["area"] = population["area"].round(2)

## 7.3. Export to Silver Layer

The `population` DataFrame is now ready to be exported to the **intermediate directory** as `population_cleaned.parquet`.

In [13]:
population.to_parquet(PATH_INTERMEDIATE / "population_cleaned.parquet")

In [14]:
df_to_verify = pd.read_parquet(PATH_INTERMEDIATE / "population_cleaned.parquet")

In [15]:
print(df_to_verify.shape, df_to_verify.dtypes.to_dict())
df_to_verify.head()

(638, 5) {'refnis': dtype('int64'), 'place_of_residence': <StringDtype(na_value=nan)>, 'population': dtype('int64'), 'area': dtype('float64'), 'population_density': dtype('float64')}


,refnis,place_of_residence,population,area,population_density
0,1000,Belgique,11431406,30689.37,372.49
1,2000,Région flamande,6589069,13625.55,483.58
2,3000,Région wallonne,3633795,16901.40,215.00
3,4000,Région de Bruxelles-Capitale,1208542,162.42,7440.85
4,10000,Province d’Anvers,1857986,2876.12,646.00
